# Feature Engineering — Home Credit Default Risk

This notebook transforms the raw `application_train.csv` file into a modeling-ready dataset: dropping sparse columns, converting Home Credit's `DAYS_*` fields into interpretable ages/tenures, engineering ratio and aggregate features, encoding categoricals, and imputing remaining missing values. The result is saved to `../data/processed/train_processed.csv` for use in model training.

This mirrors the reusable pipeline in `src/preprocess.py`, broken out step by step for inspection.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Load Data

In [ ]:
df = pd.read_csv("../data/raw/application_train.csv")
print(f"Loaded shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()


## 3. Drop Columns with >50% Missing Values

In [ ]:
missing_pct = df.isnull().mean()
threshold = 0.5
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >{threshold*100:.0f}% missing values")
df = df.drop(columns=cols_to_drop)
print(f"Shape after drop: {df.shape}")


## 4. Transform DAYS Columns

Home Credit stores several time fields as negative day counts relative to the application date. Converting them to positive, human-readable years makes them far more interpretable for both modeling and explainability.

In [ ]:
# DAYS_BIRTH -> AGE_YEARS
if "DAYS_BIRTH" in df.columns:
    df["AGE_YEARS"] = (-df["DAYS_BIRTH"]) / 365
    df = df.drop(columns=["DAYS_BIRTH"])

# DAYS_EMPLOYED -> YEARS_EMPLOYED, with an anomaly flag
# (Home Credit uses 365243 as a sentinel for "not employed" / pensioners)
if "DAYS_EMPLOYED" in df.columns:
    df["EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] > 0).astype(int)
    df["YEARS_EMPLOYED"] = (-df["DAYS_EMPLOYED"].clip(upper=0)) / 365
    df = df.drop(columns=["DAYS_EMPLOYED"])

# Remaining DAYS_* columns -> YEARS_*
for col in ["DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
    if col in df.columns:
        df[col.replace("DAYS_", "YEARS_")] = (-df[col]) / 365
        df = df.drop(columns=[col])

print("DAYS_* columns transformed. New columns:",
      [c for c in df.columns if c.startswith(("AGE_", "YEARS_", "EMPLOYED_"))])


## 5. Create Derived Features

Ratio and aggregate features that are commonly predictive of default risk: affordability ratios, income-per-person, implied loan term, and summary statistics across the three `EXT_SOURCE_*` credit bureau scores.

In [ ]:
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + 1)
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + 1)
df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"] / (df["AMT_GOODS_PRICE"] + 1)
df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"] + 1)
df["LOAN_TERM_MONTHS"] = df["AMT_CREDIT"] / (df["AMT_ANNUITY"] + 1)

ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df.columns]
if ext_cols:
    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

derived_cols = ["ANNUITY_INCOME_RATIO", "CREDIT_INCOME_RATIO", "CREDIT_GOODS_RATIO",
                "INCOME_PER_PERSON", "LOAN_TERM_MONTHS", "EXT_SOURCE_MEAN",
                "EXT_SOURCE_MIN", "EXT_SOURCE_STD"]
print("Derived features created:", derived_cols)
df[derived_cols].describe()


## 6. Encode Categorical Features

Binary categoricals (two unique values) are label-encoded; higher-cardinality categoricals are one-hot encoded.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"{len(cat_cols)} categorical columns to encode: {cat_cols}")

binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
multi_cols = [c for c in cat_cols if df[c].nunique() > 2]

print(f"Label encoding {len(binary_cols)} binary columns: {binary_cols}")
for col in binary_cols:
    df[col] = pd.factorize(df[col])[0]

print(f"One-hot encoding {len(multi_cols)} multi-class columns: {multi_cols}")
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

print(f"Shape after encoding: {df.shape}")


## 7. Median Impute Remaining Missing Values

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
n_missing_before = df[numeric_cols].isnull().sum().sum()

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

n_missing_after = df[numeric_cols].isnull().sum().sum()
print(f"Missing values before imputation: {n_missing_before:,}")
print(f"Missing values after imputation: {n_missing_after:,}")


## 8. Final Shape and Target Distribution

In [ ]:
print(f"Final processed shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("\nTarget distribution:")
print(df["TARGET"].value_counts())
print(f"\nDefault rate: {df['TARGET'].mean() * 100:.2f}%")


## 9. Save Processed Data

In [ ]:
output_path = f"{OUTPUT_DIR}/train_processed.csv"
df.to_csv(output_path, index=False)
print(f"Saved processed dataset to {output_path}")
print(f"Final shape: {df.shape}")
